# Lecture 5 Demo: Quantum Data Encoding, Overlaps, and Kernels

Computational companion notebook for Lecture 5.

Convention reminder: each qubit starts in $\lvert 0\rangle$ by default.

This notebook follows the lecture order:

1. basis, amplitude, and angle encoding,
2. exact overlaps and one-qubit kernels,
3. kernel matrices via the adjoint trick,
4. entangling feature maps,
5. a short PennyLane comparison,
6. a re-uploading preview.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector

import pennylane as qml

print(f"Qiskit version: {__import__('qiskit').__version__}")
print(f"PennyLane version: {qml.__version__}")

## Demo 1. Basis, amplitude, and angle encoding as explicit states

We first build small statevectors for the three main encoding strategies.

This is useful because later the kernel will compare these states through overlaps.

In [ ]:
basis_state = Statevector.from_label("01")

amp_raw = np.array([0.3, 0.7, 0.5, 0.1], dtype=float)
amp_state = amp_raw / np.linalg.norm(amp_raw)

qc_angle = QuantumCircuit(2)
qc_angle.ry(np.pi / 4, 0)
qc_angle.ry(np.pi / 3, 1)
angle_state = Statevector.from_instruction(qc_angle)

print("Basis encoding |01>:", np.round(basis_state.data, 6))
print()
print("Amplitude-encoded normalized vector:", np.round(amp_state, 6))
print("Norm:", np.linalg.norm(amp_state))
print()
print("Angle-encoded two-qubit statevector:")
print(np.round(angle_state.data, 6))

## Demo 2. Exact overlap, fidelity, and the one-qubit closed-form kernel

For a one-qubit angle encoding
$$
\lvert\phi(x)\rangle = R_y(x)\lvert 0\rangle,
$$
we can compare a numerical statevector overlap with the closed-form kernel
$$
K_Q(x,y)=\cos^2\left(\frac{x-y}{2}\right).
$$

In [ ]:
def ry_state(theta: float) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    return Statevector.from_instruction(qc).data


theta_a = 0.9
theta_b = 1.4

sv_a = ry_state(theta_a)
sv_b = ry_state(theta_b)
overlap = np.vdot(sv_a, sv_b)
fidelity = float(np.abs(overlap) ** 2)
kernel_formula = float(np.cos((theta_a - theta_b) / 2) ** 2)

print(f"<phi(x)|phi(y)> = {overlap:.6f}")
print(f"Fidelity from statevectors: {fidelity:.6f}")
print(f"Closed-form kernel value:   {kernel_formula:.6f}")

## Demo 3. Compare classical distance, classical similarity, and quantum fidelity

Even for one-dimensional inputs, distance and similarity are not the same object.

We compare:
- Euclidean distance $\lvert x-y\rvert$
- an RBF similarity $e^{-\gamma (x-y)^2}$
- the one-qubit quantum kernel $\cos^2((x-y)/2)$

In [ ]:
points = np.array([0.1, 0.8, 1.6])
gamma = 0.7

rows = []
for i in range(len(points)):
    for j in range(i + 1, len(points)):
        x = points[i]
        y = points[j]
        dist = abs(x - y)
        k_rbf = np.exp(-gamma * (x - y) ** 2)
        k_quantum = np.cos((x - y) / 2) ** 2
        rows.append((x, y, dist, k_rbf, k_quantum))

for x, y, dist, k_rbf, k_quantum in rows:
    print(
        f"x={x:.1f}, y={y:.1f} | distance={dist:.3f} | "
        f"RBF={k_rbf:.6f} | K_Q={k_quantum:.6f}"
    )

## Demo 4. Adjoint trick and SWAP test

For
$$
\lvert\phi(x)\rangle = U(x)\lvert 0\rangle,
\qquad
K_Q(x,y)=\left|\langle 0\vert U(x)^\dagger U(y)\vert 0\rangle\right|^2.
$$

The adjoint trick turns the kernel into a projection probability. We also compare with a SWAP-test estimate for one pair.

In [ ]:
sampler = StatevectorSampler()


def adjoint_kernel_prob0_exact(x: float, y: float) -> float:
    qc = QuantumCircuit(1)
    qc.ry(y, 0)
    qc.ry(-x, 0)
    probs = Statevector.from_instruction(qc).probabilities()
    return float(probs[0])


def adjoint_kernel_prob0_shots(x: float, y: float, shots: int = 20000) -> float:
    qc = QuantumCircuit(1)
    qc.ry(y, 0)
    qc.ry(-x, 0)
    qc.measure_all()
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    return counts.get("0", 0) / shots


def swap_test_fidelity(theta_a: float, theta_b: float, shots: int = 20000) -> float:
    qc = QuantumCircuit(3)
    anc, qa, qb = 0, 1, 2
    qc.ry(theta_a, qa)
    qc.ry(theta_b, qb)
    qc.h(anc)
    qc.cswap(anc, qa, qb)
    qc.h(anc)
    qc.measure_all()
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    p0 = sum(v for k, v in counts.items() if k.endswith("0")) / shots
    return 2 * p0 - 1

angles = np.array([0.2, 0.8, 1.4, 2.0])
K_adj = np.array([
    [adjoint_kernel_prob0_exact(x, y) for y in angles]
    for x in angles
])

f_exact = adjoint_kernel_prob0_exact(theta_a, theta_b)
f_adj_shots = adjoint_kernel_prob0_shots(theta_a, theta_b)
f_swap_shots = swap_test_fidelity(theta_a, theta_b)

print("Adjoint-trick kernel matrix:\n", np.round(K_adj, 6))
print()
print(f"Exact kernel value:        {f_exact:.6f}")
print(f"Adjoint trick (shots):     {f_adj_shots:.6f}")
print(f"SWAP test fidelity (shots):{f_swap_shots:.6f}")

## Demo 5. A small entangling feature map changes the kernel geometry

We compare a product encoding with a simple two-qubit entangling map. The point is not that one is universally better, but that they induce different pairwise similarities.

In [ ]:
def product_feature_state(x1: float, x2: float) -> np.ndarray:
    qc = QuantumCircuit(2)
    qc.ry(x1, 0)
    qc.ry(x2, 1)
    return Statevector.from_instruction(qc).data


def entangling_feature_state(x1: float, x2: float) -> np.ndarray:
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.h(1)
    qc.rz(x1, 0)
    qc.rz(x2, 1)
    qc.cx(0, 1)
    qc.rz((np.pi - x1) * (np.pi - x2), 1)
    qc.cx(0, 1)
    return Statevector.from_instruction(qc).data


def fidelity_matrix(points_2d: np.ndarray, state_fn) -> np.ndarray:
    states = [state_fn(x1, x2) for x1, x2 in points_2d]
    return np.array([
        [np.abs(np.vdot(sa, sb)) ** 2 for sb in states]
        for sa in states
    ], dtype=float)

X2 = np.array([
    [0.2, 0.4],
    [0.5, 1.0],
    [1.2, 0.7],
    [1.8, 1.6],
])

K_product = fidelity_matrix(X2, product_feature_state)
K_ent = fidelity_matrix(X2, entangling_feature_state)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, title in zip(
    axes,
    [K_product, K_ent],
    ["Product encoding", "Entangling feature map"],
):
    im = ax.imshow(mat, vmin=0.0, vmax=1.0, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("j")
    ax.set_ylabel("i")
    ax.set_xticks(range(len(X2)))
    ax.set_yticks(range(len(X2)))
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
plt.tight_layout()
plt.show()

print("Product-kernel matrix:\n", np.round(K_product, 4))
print()
print("Entangling-kernel matrix:\n", np.round(K_ent, 4))

## Demo 6. Short PennyLane comparison

We compute the same one-qubit fidelity in PennyLane to emphasize that the kernel concept is framework-independent.

In [ ]:
dev = qml.device("default.qubit", wires=1)


@qml.qnode(dev)
def qml_ry_state(theta):
    qml.RY(theta, wires=0)
    return qml.state()

qml_state_a = np.array(qml_ry_state(theta_a), dtype=complex)
qml_state_b = np.array(qml_ry_state(theta_b), dtype=complex)
qml_fidelity = float(np.abs(np.vdot(qml_state_a, qml_state_b)) ** 2)

print(f"Qiskit fidelity:    {fidelity:.6f}")
print(f"PennyLane fidelity: {qml_fidelity:.6f}")

## Demo 7. Data re-uploading preview

We compare a single-pass one-qubit encoding with a simple re-uploading circuit. The kernel matrices differ, which is the point: repeated data injection changes the effective feature map.

In [ ]:
def single_pass_state(x: float) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(x, 0)
    return Statevector.from_instruction(qc).data


def reupload_state(x: float, theta1: float = 0.6, theta2: float = -0.4) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(x, 0)
    qc.rz(0.5 * x, 0)
    qc.ry(theta1, 0)
    qc.ry(x, 0)
    qc.rz(0.5 * x, 0)
    qc.ry(theta2, 0)
    return Statevector.from_instruction(qc).data


def kernel_matrix_1d(values: np.ndarray, state_fn) -> np.ndarray:
    states = [state_fn(v) for v in values]
    return np.array([
        [np.abs(np.vdot(sa, sb)) ** 2 for sb in states]
        for sa in states
    ], dtype=float)

values = np.linspace(0.0, 2.4, 6)
K_single = kernel_matrix_1d(values, single_pass_state)
K_reupload = kernel_matrix_1d(values, reupload_state)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, title in zip(
    axes,
    [K_single, K_reupload],
    ["Single-pass encoding", "Data re-uploading"],
):
    im = ax.imshow(mat, vmin=0.0, vmax=1.0, cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("j")
    ax.set_ylabel("i")
    ax.set_xticks(range(len(values)))
    ax.set_yticks(range(len(values)))
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
plt.tight_layout()
plt.show()

print("Single-pass kernel matrix:\n", np.round(K_single, 4))
print()
print("Re-uploading kernel matrix:\n", np.round(K_reupload, 4))

## Summary

A quantum kernel is not an abstract extra object. It is the overlap geometry induced by an encoding circuit. Changing the encoding changes the state family, and changing the state family changes the Gram matrix seen by the downstream classical learner.